In [37]:
from pathlib import Path
from loguru import logger
import pandas as pd
from datetime import datetime

Read in the file

In [38]:
import tomllib

configfile = Path("../config.toml").resolve()
with configfile.open("rb") as f:
    config = tomllib.load(f)
processed = Path("../data/processed")
datafile = processed / config["inputpath"]
if not datafile.exists():
    logger.warning(
        f"{datafile} does not exist. Maybe first run src/preprocess.py, or check the timestamp!"
    )

In [39]:
df = pd.read_csv(datafile, parse_dates=["timestamp"])
df.head()

,timestamp,author,message
0,2013-08-17 10:14:17,Family app 👨‍👨‍👧‍👦,‎Berichten en oproepen worden end-to-end vers...
1,2013-08-17 10:14:17,Loïs Vriens,‎Loïs Vriens heeft deze groep gemaakt\n
2,2023-08-16 19:50:13,Family app 👨‍👨‍👧‍👦,‎Bram Van Den Oever heeft je toegevoegd\n
3,2023-08-16 19:59:39,Loïs Vriens,‎Dit bericht is verwijderd.\n
4,2023-08-16 20:00:35,Loïs Vriens,‎ ‎GIF weggelaten\n


Sometimes, author names have a tilde in front of them, allong with some unicode. Let's clean that.

In [40]:
import re

clean_tilde = r"^~\u202f"
df["author"] = df["author"].apply(lambda x: re.sub(clean_tilde, "", x))

Let's check how many unique authors we have

In [41]:
len(df.author.unique())
df.author.unique()

array(['Family app 👨\u200d👨\u200d👧\u200d👦', 'Loïs Vriens',
       'Kris van den Oever', 'Cheryl Van Den Oever', 'Regina Veld',
       'Gijs van den Oever', 'Melissa', 'Ties Van Den Oever',
       'Bram Van Den Oever', 'Allard', 'Jos Vriens',
       'Anouk Van Den Oever-blankers', 'Wendy Van Den Oever'],
      dtype=object)

Let's add age and gender

In [42]:
ages = {
    'Loïs Vriens': 24,
    'Kris van den Oever': 30,
    'Cheryl Van Den Oever': 28,
    'Regina Veld': 54,
    'Gijs van den Oever': 34,
    'Melissa': 32,
    'Ties Van Den Oever': 33,
    'Bram Van Den Oever' : 37,
    'Allard' : 36,
    'Jos Vriens' : 60,
    'Anouk Van Den Oever-blankers' : 27,
    'Wendy Van Den Oever' : 35
}

genders = {
    'Loïs Vriens': 'f',
    'Kris van den Oever': 'm',
    'Cheryl Van Den Oever': 'f',
    'Regina Veld': 'f',
    'Gijs van den Oever': 'm',
    'Melissa': 'f',
    'Ties Van Den Oever': 'm',
    'Bram Van Den Oever' : 'm',
    'Allard' : 'm',
    'Jos Vriens' : 'm',
    'Anouk Van Den Oever-blankers' : 'm',
    'Wendy Van Den Oever' : 'f'
}

# Voeg leeftijd toe
df['age'] = df['author'].map(ages)

# Voeg geslacht toe
df['gender'] = df['author'].map(genders)

df.head()

,timestamp,author,message,age,gender
0,2013-08-17 10:14:17,Family app 👨‍👨‍👧‍👦,‎Berichten en oproepen worden end-to-end vers...,NaN,NaN
1,2013-08-17 10:14:17,Loïs Vriens,‎Loïs Vriens heeft deze groep gemaakt\n,24.0,f
2,2023-08-16 19:50:13,Family app 👨‍👨‍👧‍👦,‎Bram Van Den Oever heeft je toegevoegd\n,NaN,NaN
3,2023-08-16 19:59:39,Loïs Vriens,‎Dit bericht is verwijderd.\n,24.0,f
4,2023-08-16 20:00:35,Loïs Vriens,‎ ‎GIF weggelaten\n,24.0,f


In [43]:
import json
from wa_analyzer.humanhasher import humanize

authors = df.author.unique()
anon = {k: humanize(k) for k in authors}
# we save a reference file so we can look up the original author names if we want to
reference_file = processed / "anon_reference.json"

with open(reference_file, "w") as f:
    # invert the dictionary:
    ref = {v: k for k, v in anon.items()}
    # sort alphabetically:
    ref_sorted = {k: ref[k] for k in sorted(ref.keys())}
    # save as json:
    json.dump(ref_sorted, f)

assert len(anon) == len(authors), "you lost some authors!"

In [44]:
# set anonymous authors
df["anon_author"] = df.author.map(anon)

# drop author
df.drop(columns=["author"], inplace=True)
df

,timestamp,message,age,gender,anon_author
0,2013-08-17 10:14:17,‎Berichten en oproepen worden end-to-end vers...,NaN,NaN,furry-binturong
1,2013-08-17 10:14:17,‎Loïs Vriens heeft deze groep gemaakt\n,24.0,f,good-natured-ermine
2,2023-08-16 19:50:13,‎Bram Van Den Oever heeft je toegevoegd\n,NaN,NaN,furry-binturong
3,2023-08-16 19:59:39,‎Dit bericht is verwijderd.\n,24.0,f,good-natured-ermine
4,2023-08-16 20:00:35,‎ ‎GIF weggelaten\n,24.0,f,good-natured-ermine
...,...,...,...,...,...
9894,2025-03-16 17:16:14,"Thanks 😊\nFf tussenstop op Tenerife, crew wis...",37.0,m,crystalline-turkey
9895,2025-03-16 22:33:05,"Wij zijn weer in Nederland, net de wieltjes w...",37.0,m,crystalline-turkey
9896,2025-03-17 07:24:32,Welcome back! 💪🏻\n,34.0,m,groovy-salmon
9897,2025-03-17 15:14:03,‎ ‎afbeelding weggelaten\n,24.0,f,good-natured-ermine


In [45]:
# rename author colomn
df.rename(columns={"anon_author": "author"}, inplace=True)
df.head()

,timestamp,message,age,gender,author
0,2013-08-17 10:14:17,‎Berichten en oproepen worden end-to-end vers...,NaN,NaN,furry-binturong
1,2013-08-17 10:14:17,‎Loïs Vriens heeft deze groep gemaakt\n,24.0,f,good-natured-ermine
2,2023-08-16 19:50:13,‎Bram Van Den Oever heeft je toegevoegd\n,NaN,NaN,furry-binturong
3,2023-08-16 19:59:39,‎Dit bericht is verwijderd.\n,24.0,f,good-natured-ermine
4,2023-08-16 20:00:35,‎ ‎GIF weggelaten\n,24.0,f,good-natured-ermine


In my case, the first line is a header, saying messages are encrypted. Let's remove that. Your data might be different, so double check if you also want to remove the first line!

In [46]:
df = df.drop(index=[0])

let's check:

In [47]:
df.head()

,timestamp,message,age,gender,author
1,2013-08-17 10:14:17,‎Loïs Vriens heeft deze groep gemaakt\n,24.0,f,good-natured-ermine
2,2023-08-16 19:50:13,‎Bram Van Den Oever heeft je toegevoegd\n,NaN,NaN,furry-binturong
3,2023-08-16 19:59:39,‎Dit bericht is verwijderd.\n,24.0,f,good-natured-ermine
4,2023-08-16 20:00:35,‎ ‎GIF weggelaten\n,24.0,f,good-natured-ermine
5,2023-08-16 20:07:02,Klasse goed bezig 😁\n,30.0,m,snappy-ermine


Let's create a timestamp for a new, unique, filename.

In [48]:
now = datetime.now().strftime("%Y%m%d-%H%M%S")
output = processed / f"whatsapp-{now}.csv"
output

PosixPath('../data/processed/whatsapp-20250323-103744.csv')

Let's save the file both as a csv and as a parquet file.
Parquet has some advantages:
- its about 100x faster to read and write
- datatypes are preserved (eg the timestamp type). You will loose this in a csv file.
- file size is much smaller

The advantage of csv is that you can easily peak at the data in a text editor.

In [49]:
df.to_csv(output, index=False)
df.to_parquet(output.with_suffix(".parq"), index=False)

Now, go to `config.toml` and change the name by "current" to the parquet file you just created.
This makes it easier to use the same file everywhere, without the need to continuously retype the name if you change it.